# 07 · Monitoring thresholds — Keelpay copilot

Companion of `07_operations.md` (issue #64). It sets three monitoring thresholds **from files already in the repo**, so that each alarm has a number and a reason:

1. **Drift without labels** (layer 1, daily): population stability index (PSI) of the Qualifier outputs and of the ticket language, reference = the validation split.
2. **Quality with labels** (layer 2, weekly): size of the labelled audit sample and the error count that raises the alarm.
3. **Latency** (layer 0, live): Qualifier service-level objective from the measured CPU timings of D-q4.

All data is synthetic (`agentic-workflow-subproject/data/dataset/`, `docs/results/`). The splits were written by different models (DeepSeek for train/val, Xiaomi for the tests, `governance.md` Session 3) and the traps are a stress set: they play the role of a "shifted week" here, not of Keelpay traffic.

In [1]:
import csv, json, math
from collections import Counter
from pathlib import Path
from scipy.stats import binom

ROOT = Path.cwd()
while not (ROOT / "agentic-workflow-subproject").exists():
    ROOT = ROOT.parent
SUB = ROOT / "agentic-workflow-subproject"
DATA, RES = SUB / "data" / "dataset", SUB / "docs" / "results"

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

# The rules of the output contract, copied from ml/qualifier/src/dataset_build/schema.py
# (TicketLabel) so that this notebook runs without the sub-project environment.
import re
SUB_INTENTS = {
    "payment_status": ("delayed_credit", "missing_payment", "status_inquiry", "refund_status"),
    "dispute": ("unauthorised_fee", "chargeback", "duplicate_charge", "amount_mismatch"),
    "technical_api": ("webhook_down", "api_error", "auth_failure", "integration_question"),
    "compliance": ("kyc_update", "account_hold", "document_request", "regulatory_question"),
}
SENTINELS = ("OTHER", "UNDEFINED")
KEYS = {"intent", "sub_intent", "transaction_id", "amount", "currency", "urgency", "sentiment"}
CURRENCIES = ("EUR", "USD", "GBP", "CHF", "CAD", "SEK", "DKK", "NOK", "PLN")

def _nullable_ok(v, ok):
    return v in ("", "UNDEFINED") or ok(v)

def parse(js):
    """The seven fields when the output is valid against the contract, else None."""
    try:
        d = json.loads(js)
    except (TypeError, ValueError):
        return None
    if not isinstance(d, dict) or set(d) != KEYS or not all(isinstance(v, str) for v in d.values()):
        return None
    i, s = d["intent"], d["sub_intent"]
    ok = (
        i in (*SUB_INTENTS, *SENTINELS)
        and (s == i if i in SENTINELS else s in (*SUB_INTENTS[i], *SENTINELS))
        and d["urgency"] in ("high", "medium", "low", "UNDEFINED")
        and d["sentiment"] in ("negative", "neutral", "positive", "UNDEFINED")
        and _nullable_ok(d["transaction_id"], lambda v: re.fullmatch(r"TX-\d{6}", v))
        and _nullable_ok(d["amount"], lambda v: re.fullmatch(r"\d+\.\d{2}", v))
        and _nullable_ok(d["currency"], lambda v: v in CURRENCIES)
    )
    return d if ok else None

preds = {
    "val (reference)": read_csv(RES / "d_q4_full_run_val_predictions.csv"),
    "test_natural": read_csv(RES / "d_q4_test_natural_predictions.csv"),
    "test_traps": read_csv(RES / "d_q4_test_traps_predictions.csv"),
}
for k, rows in preds.items():
    print(f"{k:<16} {len(rows)} predictions")

val (reference)  293 predictions
test_natural     610 predictions
test_traps       380 predictions


## 1 · Drift without labels: PSI of what the Qualifier outputs

In production no label exists on the day; what the copilot *does* exist: the intent it predicts, the urgency, the share of `UNDEFINED`, the share of invalid outputs. A shift in these shares is the first sign that the traffic or the model changed.

PSI = Σ (p − q) · ln(p / q) over the categories, q = reference share, p = share of the period. Usual reading: < 0.10 stable, 0.10–0.25 watch, > 0.25 act. A floor of 0.5 % per category avoids dividing by zero.

In [2]:
def psi(ref_counts, cur_counts, floor=0.005):
    cats = set(ref_counts) | set(cur_counts)
    nr, nc = sum(ref_counts.values()), sum(cur_counts.values())
    total = 0.0
    for c in cats:
        q = max(ref_counts.get(c, 0) / nr, floor)
        p = max(cur_counts.get(c, 0) / nc, floor)
        total += (p - q) * math.log(p / q)
    return total

def field_counts(rows, field):
    out = Counter()
    for r in rows:
        d = parse(r["extracted_response"])
        out["INVALID" if d is None else str(d.get(field, "MISSING"))] += 1
    return out

def band(v):
    return "stable" if v < 0.10 else ("watch" if v < 0.25 else "ACT")

ref = preds["val (reference)"]
print(f"{'field':<10}{'period':<15}{'PSI':>7}  reading")
for field in ["intent", "urgency", "sentiment"]:
    for name in ["test_natural", "test_traps"]:
        v = psi(field_counts(ref, field), field_counts(preds[name], field))
        print(f"{field:<10}{name:<15}{v:>7.3f}  {band(v)}")

field     period             PSI  reading
intent    test_natural     0.009  stable
intent    test_traps       0.389  ACT
urgency   test_natural     0.008  stable
urgency   test_traps       0.206  watch
sentiment test_natural     0.015  stable
sentiment test_traps       0.069  stable


In [3]:
def share(rows, test):
    return sum(test(r) for r in rows) / len(rows)

def is_invalid(r):
    return parse(r["extracted_response"]) is None

def is_undefined_intent(r):
    d = parse(r["extracted_response"])
    return d is not None and d.get("intent") == "UNDEFINED"

print(f"{'period':<16}{'invalid':>9}{'intent UNDEFINED':>18}")
for name, rows in preds.items():
    print(f"{name:<16}{share(rows, is_invalid):>9.1%}{share(rows, is_undefined_intent):>18.1%}")

period            invalid  intent UNDEFINED
val (reference)      1.4%              9.6%
test_natural         2.1%              9.7%
test_traps           4.7%              5.8%


### Drift of the inputs: language mix

The same index on the ticket language (dataset column `language`), reference = train. A new language or a new merchant segment appears here before it shows in the errors.

In [4]:
lang = {name: Counter(r["language"] for r in read_csv(DATA / f"{name}.csv"))
        for name in ["train", "val", "test_natural", "test_traps"]}
for name in ["val", "test_natural", "test_traps"]:
    v = psi(lang["train"], lang[name])
    top = ", ".join(f"{k} {c / sum(lang[name].values()):.0%}" for k, c in lang[name].most_common(3))
    print(f"{name:<13} PSI {v:.3f} ({band(v)})  top: {top}")

val           PSI 0.016 (stable)  top: fr 55%, en 42%, es 2%
test_natural  PSI 0.008 (stable)  top: fr 51%, en 48%, es 1%
test_traps    PSI 0.043 (stable)  top: fr 52%, en 42%, de 2%


## 2 · Quality with labels: the weekly audit sample

A person labels a random sample of auto-routed tickets every week (the QA sample of `05_metrics.md`). Reference rate: D-q4 silent errors on `test_natural`, **0.7 per 100** (`docs/scoring_protocol.md` §8).

The alarm fires when the count of silent errors has less than a 1 % chance under the reference rate. Target: catch a **tripling** (2.1 %) at least 80 % of the time. The table shows what one week can do, then what a rolling four-week window of the same weekly sample can do.

In [5]:
P0, P_BAD = 0.007, 0.021

def alarm_threshold(n, p0=P0, false_alarm=0.01):
    k = 0
    while binom.sf(k - 1, n, p0) > false_alarm:   # P(X >= k)
        k += 1
    return k

def row(label, n):
    k = alarm_threshold(n)
    print(f"{label:>22}{n:>7}{k:>13}{binom.sf(k - 1, n, P0):>13.3%}{binom.sf(k - 1, n, P_BAD):>13.0%}")

print(f"{'window':>22}{'n':>7}{'alarm at >=':>13}{'false alarm':>13}{'catch 2.1 %':>13}")
for n in [100, 200, 400]:
    row("one week", n)
for n in [100, 150, 200]:
    row(f"4 weeks of {n}/week", 4 * n)

                window      n  alarm at >=  false alarm  catch 2.1 %
              one week    100            4       0.553%          16%
              one week    200            6       0.306%          25%
              one week    400            8       0.789%          60%
   4 weeks of 100/week    400            8       0.789%          60%
   4 weeks of 150/week    600           11       0.393%          71%
   4 weeks of 200/week    800           13       0.498%          86%


**Reading.** One week of any affordable sample is too small: 200 audited tickets catch a tripling only about one time in four. The alarm therefore reads a **rolling four-week window**: 200 labelled tickets a week (about 2.3 % of the 8,515 weekly tickets, ~1 h 40 of a person at 30 s a label) give 800 tickets, which catch a tripling most of the time with under 1 % false alarms. The cost is speed: a slow degradation is seen within about a month. Fast, severe failures are not left to this sample: urgent tickets in a wrong desk are not sampled, every Re-router correction of an urgent ticket is counted, and one is an incident (floor 0); drift (layer 1) and complaints are daily.

## 3 · Latency: the Qualifier service-level objective

Measured on CPU (8 vCPU Xeon) over the 610 tickets of `test_natural` (`docs/results/d_compact_test_natural_timings.json`).

In [6]:
t = json.loads((RES / "d_compact_test_natural_timings.json").read_text())["q4"]
print(f"D-q4 CPU: median {t['seconds_median']} s, mean {t['seconds_mean']} s, p95 {t['seconds_p95']} s, "
      f"parseable JSON {t['valid_json']}/610 (contract validity: 97.9 %, section 1)")
slo_p95 = math.ceil(t["seconds_p95"] * 1.5)
print(f"SLO: p95 <= {slo_p95} s over a rolling hour (1.5 x measured p95); "
      f"timeout {120} s (flow/config.toml) -> ticket to the Re-router as 'unavailable'")

D-q4 CPU: median 7.056 s, mean 7.412 s, p95 11.821 s, parseable JSON 610/610 (contract validity: 97.9 %, section 1)
SLO: p95 <= 18 s over a rolling hour (1.5 x measured p95); timeout 120 s (flow/config.toml) -> ticket to the Re-router as 'unavailable'


## 4 · Summary for `07_operations.md`

| Layer | Signal | Reference (this notebook) | Watch | Act |
|---|---|---|---|---|
| 0 · live | Qualifier p95 latency | 11.8 s on CPU | > 18 s over an hour | timeouts > 5 % of an hour: new tickets to L1 |
| 1 · daily, no labels | PSI of predicted intent and urgency | 0.01 on test_natural; 0.39 / 0.21 on test_traps | 0.10–0.25 | > 0.25 two days running: labelled re-evaluation |
| 1 · daily, no labels | PSI of ticket language | ≤ 0.04 on every split | 0.10–0.25 | > 0.25: check the new language before trusting the copilot on it |
| 1 · daily, no labels | Outputs invalid against the contract | 1.4 % val, 2.1 % test_natural, 4.7 % test_traps | > 3.4 % | > 5 % |
| 1 · daily, no labels | Predicted intent `UNDEFINED` (Re-router load) | 9.6 % val | > 11.6 % | > 15 % |
| 2 · weekly, labels | Silent errors, rolling 4 weeks of 200 audited tickets | 0.7 per 100 | 10–12 of 800 | ≥ 13 of 800: re-evaluation; confirmed: model rollback |
| 2 · weekly, labels | Urgent ticket in a wrong desk | 1 of 167 on test_natural | — | 1 = incident |

Numbers are starting values from synthetic data; they are re-set on the first four pilot weeks and never loosened without a written reason in `governance.md`.